In [2]:
import pandas as pd
import numpy as np
import torch

df = pd.read_csv("../data/yelp_sample_1000.csv")

eval_indices = pd.read_csv("../data/eval_indices.csv")

print("Dataset Shape:", df.shape)
print("Evaluation Indices Shape:", eval_indices.shape)
print("Columns:", eval_indices.columns.tolist())

display(eval_indices.head())

Dataset Shape: (1000, 3)
Evaluation Indices Shape: (200, 1)
Columns: ['original_index']


,original_index
0,673
1,879
2,165
3,666
4,776


In [3]:
indices = eval_indices["original_index"].to_numpy()

eval_df = df.iloc[indices].copy()

print("Evaluation Shape:", eval_df.shape)
print(eval_df["stars"].value_counts().sort_index())

Evaluation Shape: (200, 3)
stars
1    40
2    40
3    40
4    40
5    40
Name: count, dtype: int64


In [4]:
assert len(eval_df) == 200
assert eval_df.index.is_unique
assert np.array_equal(eval_df.index.to_numpy(), indices)
assert (eval_df["stars"].value_counts().sort_index() == 40).all()

print("Evaluation Data: PASSED")

Evaluation Data: PASSED


In [8]:
predictions_a = pd.read_csv("../reports/hf_evaluation_predictions.csv")

assert len(predictions_a) == 200
assert predictions_a["row_id"].is_unique
assert set(predictions_a["row_id"]) == set(eval_df.index)

aligned_predictions = predictions_a.set_index("row_id").loc[eval_df.index]

assert np.array_equal(
    aligned_predictions["actual_stars"].to_numpy(),
    eval_df["stars"].to_numpy()
)

print("Model A Alignment: PASSED")

Model A Alignment: PASSED


In [10]:
from transformers import pipeline

MODEL_B = "tabularisai/multilingual-sentiment-analysis"

classifier_b = pipeline(
    "text-classification",
    model=MODEL_B
)

print(classifier_b.model.config.id2label)

/home/pasali/anaconda3/envs/myenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 12257.50it/s]


{0: 'Very Negative', 1: 'Negative', 2: 'Neutral', 3: 'Positive', 4: 'Very Positive'}


In [11]:
def sentiment_label_to_stars(label):
    label_mapping = {
        "Very Negative": 1,
        "Negative": 2,
        "Neutral": 3,
        "Positive": 4,
        "Very Positive": 5
    }

    if label not in label_mapping:
        raise ValueError(f"Geçersiz etiket: {label}")

    return label_mapping[label]


assert sentiment_label_to_stars("Very Negative") == 1
assert sentiment_label_to_stars("Negative") == 2
assert sentiment_label_to_stars("Neutral") == 3
assert sentiment_label_to_stars("Positive") == 4
assert sentiment_label_to_stars("Very Positive") == 5

print("Label Mapping: PASSED")

Label Mapping: PASSED


In [12]:
texts = eval_df["text"].tolist()

print("Evaluation Samples:", len(texts))

assert len(texts) == 200

Evaluation Samples: 200


In [13]:
predictions_b = classifier_b(
    texts,
    truncation=True,
    max_length=256,
    batch_size=8
)

print("Model B Predictions:", len(predictions_b))

assert len(predictions_b) == 200

Model B Predictions: 200


In [14]:
print("İlk 5 tahmin:")

for prediction in predictions_b[:5]:
    print(prediction)

print("Benzersiz etiketler:")

print(set(p["label"] for p in predictions_b))

İlk 5 tahmin:
{'label': 'Negative', 'score': 0.7728865146636963}
{'label': 'Very Negative', 'score': 0.8795689940452576}
{'label': 'Negative', 'score': 0.6250442862510681}
{'label': 'Negative', 'score': 0.654024600982666}
{'label': 'Negative', 'score': 0.47649598121643066}
Benzersiz etiketler:
{'Positive', 'Very Negative', 'Negative', 'Very Positive', 'Neutral'}


In [15]:
y_true_b = eval_df["stars"].to_numpy()

y_pred_b = np.array([
    sentiment_label_to_stars(pred["label"])
    for pred in predictions_b
])

scores_b = np.array([
    pred["score"]
    for pred in predictions_b
])

print("Actual Labels Shape:", y_true_b.shape)
print("Predicted Labels Shape:", y_pred_b.shape)
print("Scores Shape:", scores_b.shape)

print("İlk 5 gerçek etiket:", y_true_b[:5])
print("İlk 5 tahmin:", y_pred_b[:5])

Actual Labels Shape: (200,)
Predicted Labels Shape: (200,)
Scores Shape: (200,)
İlk 5 gerçek etiket: [3 1 2 2 4]
İlk 5 tahmin: [2 1 2 2 2]


In [16]:
assert len(y_true_b) == 200
assert len(y_pred_b) == 200
assert len(scores_b) == 200

assert np.isin(y_true_b, [1, 2, 3, 4, 5]).all()
assert np.isin(y_pred_b, [1, 2, 3, 4, 5]).all()

assert np.isfinite(scores_b).all()
assert ((scores_b >= 0) & (scores_b <= 1)).all()

print("Model B Prediction Validation: PASSED")

Model B Prediction Validation: PASSED


In [17]:
from pathlib import Path

model_b_df = pd.DataFrame({
    "row_id": eval_df.index,
    "actual_stars": y_true_b,
    "predicted_stars": y_pred_b,
    "score": scores_b
})

Path("../reports").mkdir(parents=True, exist_ok=True)

model_b_df.to_csv(
    "../reports/model_b_predictions.csv",
    index=False
)

print("Model B Predictions:", len(model_b_df))
display(model_b_df.head())

Model B Predictions: 200


,row_id,actual_stars,predicted_stars,score
0,673,3,2,0.772887
1,879,1,1,0.879569
2,165,2,2,0.625044
3,666,2,2,0.654025
4,776,4,2,0.476496


In [19]:
assert len(model_b_df) == 200
assert model_b_df["row_id"].is_unique

assert np.array_equal(
    aligned_predictions.index.to_numpy(),
    model_b_df["row_id"].to_numpy()
)

assert np.array_equal(
    aligned_predictions["actual_stars"].to_numpy(),
    model_b_df["actual_stars"].to_numpy()
)

print("Model A / Model B Alignment: PASSED")

saved_b = pd.read_csv("../reports/model_b_predictions.csv")

assert saved_b.shape == (200, 4)

assert np.array_equal(
    saved_b["row_id"].to_numpy(),
    model_b_df["row_id"].to_numpy()
)

assert np.array_equal(
    saved_b["predicted_stars"].to_numpy(),
    model_b_df["predicted_stars"].to_numpy()
)

print("Model B CSV Validation: PASSED")

Model A / Model B Alignment: PASSED
Model B CSV Validation: PASSED


In [20]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

def evaluate_model(y_true, y_pred):
    one_star_mask = y_true == 1

    one_star_recall = np.mean(y_pred[one_star_mask] == 1)
    one_star_error_rate = np.mean(y_pred[one_star_mask] != 1)

    assert np.isclose(one_star_error_rate, 1 - one_star_recall)

    return {
        "Evaluation Samples": len(y_true),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", labels=[1, 2, 3, 4, 5], zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", labels=[1, 2, 3, 4, 5], zero_division=0),
        "Macro-F1": f1_score(y_true, y_pred, average="macro", labels=[1, 2, 3, 4, 5], zero_division=0),
        "1-Star Recall": one_star_recall,
        "1-Star Error Rate": one_star_error_rate
    }

In [21]:
y_true_a = aligned_predictions["actual_stars"].to_numpy()
y_pred_a = aligned_predictions["predicted_stars"].to_numpy()

y_true_b = saved_b["actual_stars"].to_numpy()
y_pred_b = saved_b["predicted_stars"].to_numpy()

assert np.array_equal(
    aligned_predictions.index.to_numpy(),
    saved_b["row_id"].to_numpy()
)

assert np.array_equal(y_true_a, y_true_b)

metrics_a = evaluate_model(y_true_a, y_pred_a)
metrics_b = evaluate_model(y_true_b, y_pred_b)

comparison_df = pd.DataFrame({
    "Model A": metrics_a,
    "Model B": metrics_b
})

display(comparison_df.round(4))

,Model A,Model B
Evaluation Samples,200.0000,200.0000
Accuracy,0.5550,0.4200
Macro Precision,0.5605,0.4319
Macro Recall,0.5550,0.4200
Macro-F1,0.5532,0.3827
1-Star Recall,0.6000,0.5000
1-Star Error Rate,0.4000,0.5000


In [22]:
for model_name, metrics in [
    ("Model A", metrics_a),
    ("Model B", metrics_b)
]:
    recall = metrics["1-Star Recall"]
    error_rate = metrics["1-Star Error Rate"]

    assert np.isclose(error_rate, 1 - recall)

    print(f"{model_name} — 1-Star Recall: {recall:.4f}")
    print(f"{model_name} — 1-Star Error Rate: {error_rate:.4f}")
    print(f"{model_name} — Validation: PASSED")

Model A — 1-Star Recall: 0.6000
Model A — 1-Star Error Rate: 0.4000
Model A — Validation: PASSED
Model B — 1-Star Recall: 0.5000
Model B — 1-Star Error Rate: 0.5000
Model B — Validation: PASSED


In [23]:
import time

sample_df = eval_df.groupby("stars").sample(
    n=4,
    random_state=42
)

latency_eval = eval_df.loc[
    eval_df.index.isin(sample_df.index)
].copy()

latency_texts = latency_eval["text"].tolist()

print("Toplam yorum:", len(latency_texts))
print(latency_eval["stars"].value_counts().sort_index())

assert len(latency_texts) == 20
assert (latency_eval["stars"].value_counts().sort_index() == 4).all()

print("Latency Sample: PASSED")

Toplam yorum: 20
stars
1    4
2    4
3    4
4    4
5    4
Name: count, dtype: int64
Latency Sample: PASSED


In [ ]:
MODEL_A = "nlptown/bert-base-multilingual-uncased-sentiment"

classifier_a = pipeline(
    "text-classification",
    model=MODEL_A,
    device=classifier_b.device
)

print("Model A Device:", classifier_a.device)
print("Model B Device:", classifier_b.device)

assert classifier_a.device == classifier_b.device

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5111.10it/s]


Model A Device: cuda:0
Model B Device: cuda:0


In [25]:
import time

def synchronize_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def measure_latency(classifier, texts):
    classifier(
        texts[0],
        truncation=True,
        max_length=256,
        batch_size=1
    )

    synchronize_cuda()

    latencies = []

    for text in texts:
        synchronize_cuda()

        start = time.perf_counter()

        classifier(
            text,
            truncation=True,
            max_length=256,
            batch_size=1
        )

        synchronize_cuda()

        elapsed_ms = (time.perf_counter() - start) * 1000

        latencies.append(elapsed_ms)

    return np.array(latencies)

In [26]:
latencies_a = measure_latency(classifier_a, latency_texts)
latencies_b = measure_latency(classifier_b, latency_texts)

assert len(latencies_a) == 20
assert len(latencies_b) == 20

print("Model A Latencies:", latencies_a)
print("Model B Latencies:", latencies_b)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Model A Latencies: [29.652883 20.193231 11.577242 12.083823 10.334885  7.335941 25.606144
 81.761506  8.844274 11.368487 18.397394  8.214203 27.314554 14.220224
  6.655755 11.391113  6.756436  6.55437  10.37409   6.546154]
Model B Latencies: [ 6.898743  4.82902   4.790774  6.173492  3.872258  3.618067  5.905586
  3.509049  3.870227  3.783586  4.798275  3.638974 14.369142 11.662558
  3.653871 12.765355  5.847577  8.163069 19.171503 11.891366]


In [27]:
latency_comparison = pd.DataFrame({
    "Model A": {
        "Device": str(classifier_a.device),
        "Samples": len(latencies_a),
        "Mean Latency (ms)": np.mean(latencies_a),
        "Median Latency (ms)": np.median(latencies_a),
        "P95 Latency (ms)": np.percentile(latencies_a, 95)
    },
    "Model B": {
        "Device": str(classifier_b.device),
        "Samples": len(latencies_b),
        "Mean Latency (ms)": np.mean(latencies_b),
        "Median Latency (ms)": np.median(latencies_b),
        "P95 Latency (ms)": np.percentile(latencies_b, 95)
    }
})

display(latency_comparison)

,Model A,Model B
Device,cuda:0,cuda:0
Samples,20,20
Mean Latency (ms),16.759135,7.160625
Median Latency (ms),11.3798,5.338299
P95 Latency (ms),32.258314,14.60926


| Özellik | Model A | Model B |
|---|---|---|
| Mimari | BERT | DistilBERT |
| Lisans | MIT | CC BY-NC 4.0 |
| Ticari kullanım | Genel olarak mümkün | Kısıtlı |
| Macro-F1 | **0.5532** | 0.3827 |
| 1-Star Error Rate | **%40** | %50 |
| Ortalama Latency | 16.76 ms | **7.16 ms** |

## 1. Model Selection Matrix

| Kriter | Model A (NLP Town BERT) | Model B (TabularisAI DistilBERT) |
|---|---:|---:|
| Macro-F1 | **0.5532** | 0.3827 |
| 1-Star Error Rate | **%40** | %50 |
| Ortalama Latency | 16.76 ms | **7.16 ms** |
| P95 Latency | 32.26 ms | **14.61 ms** |
| Lisans | **MIT** | CC BY-NC 4.0 |
| Doğrudan yıldız tahmini | ✅ | ⚠️ Yaklaşık eşleme |
| Gerçek uygulamaya uygunluk | Ek testlerle değerlendirilebilir | Ticari lisans ve performans sınırlamaları var |

## 2. Karar Soruları

### Soru 1: Hangi modelin Macro-F1 değeri daha yüksek?

**Model A daha başarılıdır.**

Model A'nın Macro-F1 değeri 0.5532, Model B'nin ise 0.3827'dir. Model A, beş yıldız kategorisinin genel sınıflandırma performansında daha iyi sonuç vermiştir.

### Soru 2: Hangi model gerçek 1 yıldızlı yorumlarda daha az hata yapıyor?

**Model A daha az hata yapıyor.**

Model A'nın 1-Star Error Rate değeri %40, Model B'nin ise %50'dir.

40 gerçek 1 yıldızlı yorum içerisinde Model A 16, Model B ise 20 hata yapmıştır.

ReviewLens için olumsuz müşteri deneyimlerini doğru tespit etmek önemli olduğundan bu sonuç Model A'nın avantajıdır.

### Soru 3: Hangi model daha hızlı?

**Model B daha hızlıdır.**

Model A'nın ortalama inference süresi 16.76 ms, Model B'nin ise 7.16 ms'dir.

Model B yaklaşık 2.34 kat daha hızlıdır. Ancak bu sonuç aynı GPU üzerinde 20 yorumla yaptığımız sınırlı ölçüme aittir.

### Soru 4: Hangi modelin lisans koşulları ticari kullanım açısından daha uygun?

**Model A'nın MIT lisansı daha uygundur.**

MIT lisansı, gerekli koşullar yerine getirildiğinde ticari kullanıma genel olarak izin verir.

Model B'nin CC BY-NC 4.0 lisansı ise ticari olmayan kullanımlara yöneliktir. Ticari kullanım için hak sahibinden ayrıca izin veya uygun lisans alınması gerekebilir.

### Soru 5: Model B'nin duygu seviyelerinden yıldızlara yapılan eşlemesi ne kadar güvenilir?

**Bu eşleme kesin değildir; bir değerlendirme varsayımıdır.**

Örneğin `Negative → 2 yıldız` dönüşümünü biz belirledik.

Ancak olumsuz bir müşteri yorumu gerçekte 1 yıldızlı olabilir. Bu nedenle duygu kategorileri Yelp yıldızlarıyla her zaman birebir örtüşmeyebilir.

Bu durum Model B'nin değerlendirme sonuçlarını etkileyebilir.

### Soru 6: Bir modeli seçmek, onun gerçek uygulamada kullanılmaya hazır olduğu anlamına gelir mi?

**Hayır.**

Model seçimi, mevcut adaylar arasından en uygun modeli belirlemektir.

Gerçek bir uygulamada kullanmadan önce bağımsız test verileriyle değerlendirme, daha kapsamlı hata analizi, latency ve throughput testleri, veri güvenliği ve lisans kontrolleri yapılmalıdır.

Ayrıca modelin farklı kullanıcı yorumlarında nasıl davrandığı incelenmelidir.

## 3. Final Model Selection Decision

### Seçilen Model: Model A — NLP Town Multilingual BERT

ReviewLens projesi için **Model A'yı seçiyorum.**

Bunun temel nedeni, Model A'nın hem Macro-F1 hem de 1-Star Recall açısından daha başarılı olmasıdır.

Model B daha hızlı olmasına rağmen sınıflandırma başarısı daha düşüktür. Ayrıca Model B'nin duygu etiketlerini Yelp yıldızlarına dönüştürmek için yaklaşık bir eşleme kullanmamız gerekiyor.

Lisans açısından da Model A avantajlıdır. MIT lisansı ticari kullanım için daha uygun koşullar sunarken Model B'nin CC BY-NC 4.0 lisansı ticari kullanımı kısıtlamaktadır.

Model A'nın ortalama inference süresi 16.76 ms'dir. Model B'ye göre daha yavaş olsa da bu fark, ReviewLens'in performans gereksinimleri açısından kabul edilebilir olabilir.

**Sonuç:** Mevcut değerlendirme sonuçlarına göre Model A daha uygun adaydır. Ancak bu seçim, modelin doğrudan production ortamına alınmaya hazır olduğu anlamına gelmez. Daha kapsamlı değerlendirme ve testler gereklidir.

In [28]:
def test_label_mapping():
    model_a_mapping = {
        "1 star": 1,
        "2 stars": 2,
        "3 stars": 3,
        "4 stars": 4,
        "5 stars": 5
    }

    model_b_mapping = {
        "Very Negative": 1,
        "Negative": 2,
        "Neutral": 3,
        "Positive": 4,
        "Very Positive": 5
    }

    actual_a_labels = set(classifier_a.model.config.id2label.values())
    actual_b_labels = set(classifier_b.model.config.id2label.values())

    assert actual_a_labels == set(model_a_mapping)
    assert actual_b_labels == set(model_b_mapping)

    for label, expected in model_a_mapping.items():
        assert model_a_mapping[label] == expected

    for label, expected in model_b_mapping.items():
        assert sentiment_label_to_stars(label) == expected

    try:
        sentiment_label_to_stars("Unknown")
    except ValueError:
        pass
    else:
        raise AssertionError("Geçersiz etiket kabul edildi.")

def test_evaluation_alignment():
    assert len(aligned_predictions) == 200
    assert len(saved_b) == 200

    assert aligned_predictions.index.is_unique
    assert saved_b["row_id"].is_unique

    assert np.array_equal(
        aligned_predictions.index.to_numpy(),
        saved_b["row_id"].to_numpy()
    )

    assert np.array_equal(
        aligned_predictions.index.to_numpy(),
        eval_df.index.to_numpy()
    )

    actual_a = aligned_predictions["actual_stars"].to_numpy()
    actual_b = saved_b["actual_stars"].to_numpy()

    predicted_a = aligned_predictions["predicted_stars"].to_numpy()
    predicted_b = saved_b["predicted_stars"].to_numpy()

    assert np.array_equal(actual_a, actual_b)
    assert np.array_equal(actual_a, eval_df["stars"].to_numpy())

    assert np.isin(predicted_a, [1, 2, 3, 4, 5]).all()
    assert np.isin(predicted_b, [1, 2, 3, 4, 5]).all()

    assert np.array_equal(
        predicted_b,
        np.array([
            sentiment_label_to_stars(p["label"])
            for p in predictions_b
        ])
    )

    counts = pd.Series(actual_a).value_counts().sort_index()

    assert counts.to_dict() == {
        1: 40,
        2: 40,
        3: 40,
        4: 40,
        5: 40
    }



from sklearn.metrics import confusion_matrix

def test_metric_consistency():
    models = [
        ("Model A", aligned_predictions, metrics_a),
        ("Model B", saved_b, metrics_b)
    ]

    for name, data, metrics in models:
        actual = data["actual_stars"].to_numpy()
        predicted = data["predicted_stars"].to_numpy()

        cm = confusion_matrix(
            actual,
            predicted,
            labels=[1, 2, 3, 4, 5]
        )

        assert cm.shape == (5, 5)
        assert cm.sum() == 200

        one_star_mask = actual == 1

        assert one_star_mask.sum() == 40

        recall = np.mean(predicted[one_star_mask] == 1)
        error_rate = np.mean(predicted[one_star_mask] != 1)

        assert np.isclose(error_rate, 1 - recall)

        assert np.isclose(
            recall,
            metrics["1-Star Recall"]
        )

        assert np.isclose(
            error_rate,
            metrics["1-Star Error Rate"]
        )

In [29]:
tests = [
    ("Test 1 — Label Mapping", test_label_mapping),
    ("Test 2 — Evaluation Alignment", test_evaluation_alignment),
    ("Test 3 — Metric Consistency", test_metric_consistency)
]

for name, test in tests:
    test()
    print(f"{name}: PASSED")

Test 1 — Label Mapping: PASSED
Test 2 — Evaluation Alignment: PASSED
Test 3 — Metric Consistency: PASSED
